# Named Entity Recognition (NER) Project

## Mục tiêu
Xây dựng hệ thống nhận diện thực thể (NER) từ dữ liệu báo chí.

## Pipeline:
1. Crawl dữ liệu từ Google News
2. Tiền xử lý văn bản
3. Gán nhãn (auto + manual)
4. Huấn luyện model
5. Đánh giá & dự đoán

# 1. Thu thập dữ liệu (Crawling)
Sử dụng Google News + Newspaper3k + Chromium để lấy nội dung báo.

In [7]:
import sys
import os
 
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
 
sys.path.append(ROOT_DIR)

print("Root:", ROOT_DIR)

Root: d:\NLP\Named-Entity-Recognition


# 1. Thu thập dữ liệu (Crawling)
Sử dụng Google News + Newspaper3k + Chromium để lấy nội dung báo.

In [14]:
import pandas as pd

df = pd.read_csv("../Day1_Crawl/ket_qua_sentence.csv")
df.head() 

,Title,Sentence
0,US judge halts Trump plan to end protections f...,Summary Companies Lawsuit claims Noem hostil...
1,US judge halts Trump plan to end protections f...,"U.S. District Judge Ana Reyes in Washington, D..."
2,US judge halts Trump plan to end protections f...,The move would have taken effect on Wednesday ...
3,US judge halts Trump plan to end protections f...,The Reuters Inside Track newsletter is your es...
4,US judge halts Trump plan to end protections f...,"Reyes, who was appointed by Democratic former ..."


In [15]:

print("Dataset size:", df.shape)
print("Columns:", df.columns.tolist())

Dataset size: (3680, 2)
Columns: ['Title', 'Sentence']


# 2. Tiền xử lý dữ liệu
- Lowercase
- Remove noise (Reuters, email, ký tự đặc biệt)
- Tokenize
- Remove stopwords
- Lemmatization

In [10]:
from Day2_Preprocessing.preprocess import TextPreprocessor, DataStats
import pandas as pd

df = pd.read_csv("ket_qua_sentence.csv")

processor = TextPreprocessor()

df["clean_sentence"] = df["Sentence"].apply(processor.clean_text)

df = df[df["clean_sentence"].str.split().str.len() >= 6]
df.drop_duplicates(subset=["clean_sentence"], inplace=True)

df.to_csv("clean_news.csv", index=False)

print("Done preprocessing")

Loading data...

===== RAW DATA STATS =====
Total sentences: 3680
Unique titles: 149
Missing values: 0

Cleaning text...

Removed sentences: 567

===== CLEAN DATA STATS =====
Clean samples: 3113
Avg sentence length: 15.76
Max length: 235
Min length: 6

Top 10 frequent words:
said: 744
epstein: 324
year: 295
trump: 288
would: 205
last: 201
also: 177
reuters: 166
new: 165
monday: 157

✅ DONE PREPROCESS
Saved clean data to: clean_news.csv
Saved stats to: stats.txt
Done preprocessing


In [11]:
DataStats.raw_stats(df)
DataStats.clean_stats(df, "clean_sentence")


===== RAW DATA STATS =====
Total sentences: 3113
Unique titles: 146
Missing values: 0

===== CLEAN DATA STATS =====
Clean samples: 3113
Avg sentence length: 15.76
Max length: 235
Min length: 6

Top 10 frequent words:
said: 744
epstein: 324
year: 295
trump: 288
would: 205
last: 201
also: 177
reuters: 166
new: 165
monday: 157


(np.float64(15.756826212656602),
 [('said', 744),
  ('epstein', 324),
  ('year', 295),
  ('trump', 288),
  ('would', 205),
  ('last', 201),
  ('also', 177),
  ('reuters', 166),
  ('new', 165),
  ('monday', 157)])

# 3. Gán nhãn dữ liệu
Sử dụng:
- Transformers (dslim/bert-base-NER)
- Rule-based (DATE, MONEY, CARDINAL)

In [17]:
df = pd.read_csv("manual_labeled.csv")
print(df.head(2))
print("Total samples:", len(df))

                                              tokens  \
0  ['Summary', 'Companies', 'Lawsuit', 'claims', ...   
1  ['U', '.', 'S', '.', 'District', 'Judge', 'Ana...   

                                              labels  
0  ['O', 'O', 'O', 'O', 'B-PERSON', 'O', 'O', 'B-...  
1  ['B-GPE', 'I-GPE', 'I-GPE', 'I-GPE', 'O', 'O',...  
Total samples: 3113


# 4. Mã hoá dữ liệu
- Convert token → id
- Convert label → id
- Padding về cùng độ dài

In [ ]:
import pickle

with open("ner_dataset.pkl", "rb") as f:
    data = pickle.load(f)

X = data["X"]
y = data["y"]

print(X.shape, y.shape)

# 5. Huấn luyện mô hình
Model sử dụng:
- DistilBERT (transformers)
- Token classification

In [ ]:
from train_ner import *

In [ ]:
print("Model: distilbert-base-uncased")
print("Epochs: 5")
print("Batch size: 16")

# 6. Đánh giá mô hình
Sử dụng:
- Precision
- Recall
- F1-score

In [ ]:
from seqeval.metrics import classification_report


# 7. Demo dự đoán

In [ ]:
from inference import predict_ner

predict_ner("Apple CEO Tim Cook visited Vietnam in 2024.")

# 8. Kết luận

## Kết quả đạt được
- Xây dựng pipeline NER hoàn chỉnh
- Crawl + preprocess + labeling + training

## Hạn chế
- Dataset còn nhỏ
- Label chưa chính xác hoàn toàn

## Hướng phát triển
- Fine-tune tốt hơn
- Tăng dữ liệu
- Dùng PhoBERT cho tiếng Việt